In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
import numpy as np
import math
import random
import matplotlib.pyplot as plt

print("📌 Imports loaded.")


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1-a)))


In [ ]:
def ground_speed_to_latlon_velocity(gspeed, track_deg, lat):
    R = 6371e3
    track = math.radians(track_deg)

    v_north = gspeed * math.cos(track)
    v_east  = gspeed * math.sin(track)

    dlat = (v_north / R) * (180 / math.pi)
    dlon = (v_east  / (R * math.cos(math.radians(lat)))) * (180 / math.pi)

    return dlat, dlon


In [ ]:
def wind_to_latlon_velocity(wind_speed_ms, wind_dir_deg, lat):
    """
    Wind_dir_deg = direction FROM which the wind blows.
    Convert to TO direction by adding 180°.
    """
    R = 6371e3
    wd_to = math.radians(wind_dir_deg + 180)

    v_north = wind_speed_ms * math.cos(wd_to)
    v_east  = wind_speed_ms * math.sin(wd_to)

    dlat = (v_north / R) * (180 / math.pi)
    dlon = (v_east  / (R * math.cos(math.radians(lat)))) * (180 / math.pi)

    return dlat, dlon


In [ ]:
class KalmanFilterWindCV:
    def __init__(self, dt=1.0):
        self.dt = dt

        # State transition (CV model)
        self.F = np.eye(6)
        self.F[0,3] = dt
        self.F[1,4] = dt
        self.F[2,5] = dt

        # Measurement model: observe (lat, lon, alt)
        self.H = np.zeros((3,6))
        self.H[0,0] = 1
        self.H[1,1] = 1
        self.H[2,2] = 1

        # Noise matrices
        self.Q = np.eye(6) * 1e-4
        self.R = np.eye(3) * 1e-3

        # State & covariance
        self.x = np.zeros((6,1))
        self.P = np.eye(6)

    def initialize(self, lat, lon, alt, gspeed, track, vspeed):
        v_lat, v_lon = ground_speed_to_latlon_velocity(gspeed, track, lat)
        v_alt = vspeed / 60.0   # ft/min → ft/s approx

        self.x = np.array([
            [lat], [lon], [alt],
            [v_lat], [v_lon], [v_alt]
        ], dtype=float)

        self.P = np.eye(6)

    def predict(self, wind_lat=0, wind_lon=0):
        # normal CV prediction
        self.x = self.F @ self.x

        # add wind drift
        self.x[0,0] += wind_lat * self.dt
        self.x[1,0] += wind_lon * self.dt

        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, z):
        z = z.reshape((3,1))
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        self.P = (np.eye(6) - K @ self.H) @ self.P

    def get_state(self):
        return self.x.flatten()


In [ ]:
with open("../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r") as f:
    flights_data = json.load(f)

print(f"📌 Loaded {len(flights_data)} flights.")


In [ ]:
def extract_phase_sequences(flights_data, phase):
    seqs = []
    for flight in flights_data:
        phase_seq = flight.get(phase, [])
        seqs.append(phase_seq)
    print(f"📌 {phase} sequences:", len(seqs))
    return seqs

takeoff_sequences  = extract_phase_sequences(flights_data, "take_off")
cruise_sequences   = extract_phase_sequences(flights_data, "cruising")
landing_sequences  = extract_phase_sequences(flights_data, "landing")


In [ ]:
def kalman_predict_phase(seq):
    """
    seq = list of dicts for one phase.
    Returns KF predictions (second half) and ground truth.
    """

    mat = np.array([
        [
            p["lat"], p["lon"], p["altitude_ft"],
            p["gspeed"], p["vspeed"], p["track"],
            p["wind_speed_10000m_ms"], p["wind_dir_10000m_d"]
        ]
        for p in seq
    ], dtype=float)

    T = len(mat)
    mid = T // 2
    obs = mat[:mid]
    future = mat[mid:]

    # Initialize KF with first observation
    lat0, lon0, alt0, gs0, vs0, track0, _, _ = obs[0]
    kf = KalmanFilterWindCV(dt=1.0)
    kf.initialize(lat0, lon0, alt0, gs0, track0, vs0)

    # Feed KF with observed data
    for i in range(1, mid):
        lat, lon, alt, gs, vs, track, w_spd, w_dir = obs[i]
        wind_lat, wind_lon = wind_to_latlon_velocity(w_spd, w_dir, lat)

        kf.predict(wind_lat, wind_lon)
        kf.update(np.array([lat, lon, alt]))

    # Predict future without updating
    preds = []
    for i in range(len(future)):
        lat, lon, alt, gs, vs, track, w_spd, w_dir = future[i]
        wind_lat, wind_lon = wind_to_latlon_velocity(w_spd, w_dir, lat)

        kf.predict(wind_lat, wind_lon)
        preds.append(kf.get_state()[:3])

    preds = np.array(preds)
    gt = future[:, :3]

    return preds, gt


In [ ]:
def get_phase_sequence(flights_data, flight_id, phase):
    for flight in flights_data:
        if flight["flight_metadata"]["fr24_id"] == flight_id:
            return flight.get(phase, None)
    return None


In [ ]:
import numpy as np

def to_json_safe(arr):
    """Convert numpy arrays to plain Python floats inside lists."""
    if isinstance(arr, np.ndarray):
        return arr.astype(float).tolist()
    return arr


In [ ]:
import json, os

def save_kalman_results(json_path, phase_name, observed, ground_truth, predicted, mean_error):
    """Save KF results for one phase into results_KALMAN.json"""

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            results = json.load(f)
    else:
        results = {}

    results[phase_name] = {
        "observed": to_json_safe(observed),
        "ground_truth": to_json_safe(ground_truth),
        "predicted": to_json_safe(predicted),
        "mean_haversine_error": float(mean_error)
    }

    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)

    print(f"💾 Saved KF results for '{phase_name}' → {json_path}")


In [ ]:
def run_kalman_experiments(flights_data, json_path="results_KALMAN.json"):

    EXPERIMENTS = [
        ("39ba2570", "take_off"),
        ("39ba2570", "cruising"),
        ("39ba2570", "landing"),
    ]

    for flight_id, phase_raw in EXPERIMENTS:

        print("\n===========================================")
        print(f"📌 Running Kalman Filter — {phase_raw.upper()}")
        print("===========================================")

        # Load the raw sequence
        seq = get_phase_sequence(flights_data, flight_id, phase_raw)
        if seq is None or len(seq) < 4:
            print(f"⚠️ Sequence missing or too short for phase {phase_raw}")
            continue

        # Convert raw dicts → KF-friendly input
        # Also prepare Nx3 matrix [lat, lon, minutes_since_start]
        mat = np.array([
            [p["lat"], p["lon"], p["minutes_since_start"]]
            for p in seq
        ], dtype=float)

        T = len(mat)
        mid = T // 2

        observed     = mat[:mid]   # Nx3
        ground_truth = mat[mid:]   # Nx3

        # Run Kalman Filter on full phase
        preds_xy, gt_xy = kalman_predict_phase(seq)

        # KF returns only lat/lon. We append the correct time values.
        future_times = ground_truth[:, 2]
        predicted = np.hstack([preds_xy, future_times.reshape(-1, 1)])

        # Compute mean haversine error
        errors = [
            haversine(ground_truth[i][0], ground_truth[i][1],
                      predicted[i][0], predicted[i][1])
            for i in range(len(predicted))
        ]
        mean_error = float(np.mean(errors))

        # Save results
        save_kalman_results(
            json_path=json_path,
            phase_name=phase_raw,  # keeps naming consistent with input JSON
            observed=observed,
            ground_truth=ground_truth,
            predicted=predicted,
            mean_error=mean_error
        )

        print(f"📌 Mean error ({phase_raw}): {mean_error:.2f} m")


In [ ]:
with open("data/flight_data_with_minutes_since_start.json", "r") as f:
    flights_data = json.load(f)

run_kalman_experiments(flights_data)


In [ ]:
def evaluate_and_plot_phase(sequences, phase_name):
    print("\n=========================================")
    print(f"📌 WIND-AWARE KALMAN FILTER — {phase_name.upper()}")
    print("=========================================")

    # pick random example sequence for plotting
    seq_example = random.choice(sequences)

    errors = []
    for seq in sequences:
        try:
            preds, gt = kalman_predict_phase(seq)
        except:
            continue

        err = np.mean([
            haversine(gt[i][0], gt[i][1], preds[i][0], preds[i][1])
            for i in range(len(preds))
        ])
        errors.append(err)

    print(f"📌 Avg phase error = {np.mean(errors):.1f} meters")

    # ---------------- PLOT EXAMPLE ----------------
    preds, gt = kalman_predict_phase(seq_example)

    T = len(seq_example)
    mid = T//2

    hist = seq_example[:mid]
    hist_xy = np.array([[p["lat"], p["lon"]] for p in hist])
    gt_xy   = gt[:, :2]
    pr_xy   = preds[:, :2]

    plt.figure(figsize=(8,6))
    plt.scatter(hist_xy[:,1], hist_xy[:,0], color="blue", s=15, label="Observed (1st half)")
    plt.plot(gt_xy[:,1], gt_xy[:,0], color="green", label="Ground truth (2nd half)", linewidth=2)
    plt.plot(pr_xy[:,1], pr_xy[:,0], color="red", label="KF prediction", linewidth=2)

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"Wind-Aware KF Prediction — {phase_name.capitalize()}")
    plt.legend()
    plt.grid(True)
    plt.show()

    return np.mean(errors)


In [ ]:
err_takeoff = evaluate_and_plot_phase(takeoff_sequences, "takeoff")


In [ ]:
err_cruise  = evaluate_and_plot_phase(cruise_sequences,  "cruising")

In [ ]:
err_landing = evaluate_and_plot_phase(landing_sequences, "landing")

In [ ]:
print("\n================ FINAL ERRORS ================")
print(f"Takeoff KF error:  {err_takeoff:.1f} m")
print(f"Cruising KF error: {err_cruise:.1f} m")
print(f"Landing KF error:  {err_landing:.1f} m")

HERE WE WILL CALCULATE THE MEAN HAVERSINE ERROR OVER OUR 100 FLIGHT SAMPLE FOR THE 3 DIFFERENT FLIGHT PHASES

In [ ]:
import json

# Path to your JSON file
input_file = "../data/CDG-FCO/RESULTS/test_sample.json"

with open(input_file, "r", encoding="utf-8") as f:
    test_sample = json.load(f)


In [ ]:
def run_kalman_experiments(flights_data, json_path="../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_KALMAN.json"):

    EXPERIMENTS = []

    errors_takeoff = []
    errors_cruising = []
    errors_landing = []

    for phase, flight_ids in test_sample.items():
        for flight_id in flight_ids:
            EXPERIMENTS.append((flight_id, phase))

    for flight_id, phase_raw in EXPERIMENTS:
        # print("\n===========================================")
        # print(f"📌 Running Kalman Filter — {phase_raw.upper()}")
        # print("===========================================")

        # Load the raw sequence
        seq = get_phase_sequence(flights_data, flight_id, phase_raw)
        if seq is None or len(seq) < 4:
            # print(f"⚠️ Sequence missing or too short for phase {phase_raw}")
            continue

        # Convert raw dicts → KF-friendly input
        # Also prepare Nx3 matrix [lat, lon, minutes_since_start]
        mat = np.array([
            [p["lat"], p["lon"], p["minutes_since_start"]]
            for p in seq
        ], dtype=float)

        T = len(mat)
        mid = T // 2
        ground_truth = mat[mid:]   # Nx3
        # Run Kalman Filter on full phase
        preds_xy, gt_xy = kalman_predict_phase(seq)

        # KF returns only lat/lon. We append the correct time values.
        future_times = ground_truth[:, 2]
        predicted = np.hstack([preds_xy, future_times.reshape(-1, 1)])

        # Compute mean haversine error
        errors = [
            haversine(ground_truth[i][0], ground_truth[i][1],
                    predicted[i][0], predicted[i][1])
            for i in range(len(predicted))
        ]
        mean_error = float(np.mean(errors))

        if phase_raw == "take_off":
            errors_takeoff.append(round(mean_error/1000, 2))

        if phase_raw == "cruising":
            errors_cruising.append(round(mean_error/1000, 2))

        if phase_raw == "landing":
            errors_landing.append(round(mean_error/1000, 2))


    results = {"KALMAN": {"take_off": errors_takeoff, "cruising": errors_cruising, "landing": errors_landing}}

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"✅ Saved to {json_path}")


In [ ]:
run_kalman_experiments(flights_data)